# Proyecto #1: Biodiversity at Scale
## Parte 1: Exploración y Preparación del Dataset
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
1. Analizar la estructura taxonómica y distribución del dataset iNaturalist 2021.
2. Generar una **muestra estratégica aleatoria y reproducible** (50 especies, semilla fija `seed=42`) adaptada a un entorno de cómputo eficiente (4GB VRAM).
3. Construir el pipeline de datos PyTorch (`Dataset`, transformaciones, `DataLoader` deterministas).
4. Discutir cualitativa y cuantitativamente las dificultades inherentes a la clasificación *fine-grained* de biodiversidad.


In [ ]:
import sys
from pathlib import Path

# Agregar raíz del proyecto al path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
import matplotlib.pyplot as plt
from src.utils.seed import seed_everything
from src.data.dataset import INatDataset, SyntheticINatDataset
from src.data.sampler import create_strategic_subset
from src.data.transforms import get_transforms
from src.data.dataloader import build_dataloaders

# 100% Reproducibilidad
SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | CUDA Available: {torch.cuda.is_available()}")


### 1. Configuración de Rutas del Dataset
Define las rutas a las imágenes y anotaciones JSON de iNaturalist 2021 Mini.
Si los archivos reales aún no se encuentran descargados, el pipeline conmuta automáticamente a un generador sintético para validar todo el flujo de forma offline.


In [ ]:
DATA_ROOT = ROOT_DIR.parent / "recursos"
TRAIN_JSON = DATA_ROOT / "train_mini.json"
VAL_JSON = DATA_ROOT / "val.json"
TRAIN_IMAGES = DATA_ROOT / "train_mini"
VAL_IMAGES = DATA_ROOT / "val"

USE_SYNTHETIC = not (TRAIN_JSON.exists() and TRAIN_IMAGES.exists())
if USE_SYNTHETIC:
    print("[!] Nota: Archivos JSON de iNaturalist no encontrados en 'recursos/'.")
    print("[*] Conmutando a modo sintético para demostración de pipeline y pruebas de reproducibilidad.")
else:
    print("[✓] Dataset iNaturalist 2021 Mini detectado correctamente.")


### 2. Carga y Subsetting Estratégico (50 Clases)
Para garantizar la viabilidad experimental con 4GB de VRAM y cumplir con la Sección 17 de la rúbrica, seleccionamos una muestra estratégica aleatoria con semilla fija `seed=42`:
- **50 clases taxonómicas**
- **35 imágenes de entrenamiento por especie** (1,750 imágenes)
- **10 imágenes de validación por especie** (500 imágenes)


In [ ]:
N_CLASSES = 50
TRAIN_PER_CLASS = 35
VAL_PER_CLASS = 10
IMG_SIZE = 224

train_transform = get_transforms(split="train", img_size=IMG_SIZE, aug_mode="standard")
val_transform = get_transforms(split="val", img_size=IMG_SIZE, aug_mode="none")

if USE_SYNTHETIC:
    train_dataset = SyntheticINatDataset(num_samples=N_CLASSES * TRAIN_PER_CLASS, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED)
    val_dataset = SyntheticINatDataset(num_samples=N_CLASSES * VAL_PER_CLASS, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED+1)
else:
    train_full = INatDataset(TRAIN_JSON, TRAIN_IMAGES, transform=train_transform)
    val_full = INatDataset(VAL_JSON, VAL_IMAGES, transform=val_transform, category_to_label=train_full.category_to_label)
    
    indices, selected_classes, label_map = create_strategic_subset(
        train_full, n_classes=N_CLASSES, per_class=TRAIN_PER_CLASS, seed=SEED,
        output_manifest=str(ROOT_DIR / "configs" / "strategic_subset_seed42.json")
    )
    val_indices, _, _ = create_strategic_subset(
        val_full, n_classes=N_CLASSES, per_class=VAL_PER_CLASS, seed=SEED
    )
    from src.data.dataset import RemappedSubset
    train_dataset = RemappedSubset(train_full, indices, label_map, transform=train_transform)
    val_dataset = RemappedSubset(val_full, val_indices, label_map, transform=val_transform)

print(f"Muestra estratificada cargada con éxito:")
print(f"  - Especies seleccionadas: {N_CLASSES}")
print(f"  - Imágenes de Train: {len(train_dataset)}")
print(f"  - Imágenes de Validación: {len(val_dataset)}")


### 3. DataLoaders y Verificación de Batches
Construimos los DataLoaders con `seed_worker`, `pin_memory` y multi-threading.


In [ ]:
BATCH_SIZE = 32
train_loader, val_loader = build_dataloaders(
    train_dataset, val_dataset, batch_size=BATCH_SIZE, num_workers=2, seed=SEED
)

images, labels = next(iter(train_loader))
print(f"Batch de entrenamiento: Tensor {images.shape} | Labels: {labels.shape}")
print(f"Rango de valores de pixel (normalizados ImageNet): min={images.min():.2f}, max={images.max():.2f}")


### 4. Visualización de Ejemplos y Dificultades del Problema
Inspeccionamos muestras de las imágenes del batch.


In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(8):
    img = images[i].permute(1, 2, 0).cpu().numpy()
    img = std * img + mean  # Desnormalizar
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(f"Clase ID: {labels[i].item()}", fontsize=11)
    axes[i].axis("off")

plt.suptitle("Muestras de Entrada con Data Augmentation (iNaturalist)", fontsize=13)
plt.tight_layout()
plt.show()


### 5. Discusión: Retos de Fine-Grained Classification para Deep Learning
A partir de la exploración del dataset, identificamos los 3 retos principales solicitados en la rúbrica:
1. **Alta similitud inter-especie**: Especies congenéricas comparten anatomía, coloración y patrones generales; los rasgos discriminativos residen en detalles muy localizados (e.g., marcas alares, forma del sépalo).
2. **Gran variabilidad intra-especie**: Una misma especie exhibe transformaciones morfológicas por estadio de desarrollo (e.g., oruga vs mariposa), dimorfismo sexual y variaciones estacionales de follaje o pelaje.
3. **Fondos y condiciones de captura no controladas**: Oclusiones parciales por vegetación, iluminación variable, sombras, y escala diminuta del organismo respecto al encuadre general.
